In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, PauliGate, UnitaryGate
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp

# 1. Define the 4x4 1D Poisson finite-difference matrix [-1, 2, -1]
A_poisson = np.array([
    [ 2, -1,  0,  0],
    [-1,  2, -1,  0],
    [ 0, -1,  2, -1],
    [ 0,  0, -1,  2]
], dtype=complex)

# 2. Automatically decompose A_poisson into Pauli strings using Qiskit!
pauli_decomp = SparsePauliOp.from_operator(A_poisson)
print("Automatic Pauli LCU Decomposition of 1D Poisson Matrix:")
for pauli_str, coeff in zip(pauli_decomp.paulis.to_labels(), pauli_decomp.coeffs):
    print(f"  {coeff.real:+.2f} * {pauli_str}")

coeffs = pauli_decomp.coeffs.real
pauli_labels = pauli_decomp.paulis.to_labels() # ['II', 'IX', 'XX', 'YY']

# 3. Compute 1-norm lambda and PREPARE amplitudes
lam = np.sum(np.abs(coeffs))
print(f"\n1-Norm (lambda): {lam}")

prep_amps = np.sqrt(np.abs(coeffs) / lam)
prep = StatePreparation(prep_amps, label="PREP")
prep_dag = prep.inverse()
prep_dag.label = "PREP†"

# 4. Build 4-qubit circuit:
# q0, q1 : 2 ancilla (selection) qubits
# q2, q3 : 2 system qubits (since A_poisson is 4x4 = 2^2 x 2^2)
qc = QuantumCircuit(4)

# --- STAGE 1: PREPARE on ancilla [0, 1] ---
qc.append(prep, [0, 1])

# --- STAGE 2: SELECT on ancilla [0, 1] and system [2, 3] ---
for k, (c_k, p_label) in enumerate(zip(coeffs, pauli_labels)):
    b_str = format(k, '02b')
    sign = np.sign(c_k)
    
    # Skip +II since Identity does nothing
    if p_label == 'II' and sign > 0:
        continue
        
    # Build the 4x4 signed Pauli matrix for system qubits [q2, q3]
    p_gate = PauliGate(p_label)
    signed_matrix = sign * Operator(p_gate).data
    gate_label = f"{'-' if sign < 0 else ''}{p_label}"
    
    signed_gate = UnitaryGate(signed_matrix, label=gate_label)
    # Control on ancilla [0, 1], target on system [2, 3]
    qc.append(signed_gate.control(2, ctrl_state=b_str), [0, 1, 2, 3])

# --- STAGE 3: PREPARE^dagger on ancilla [0, 1] ---
qc.append(prep_dag, [0, 1])

print("\nLCU Circuit for 4x4 Poisson Matrix:")
print(qc.draw(fold=-1))

# --- STAGE 4: VERIFY THE 4x4 BLOCK ENCODING INSIDE THE 16x16 UNITARY ---
U_full = Operator(qc).data  # 16x16 unitary matrix

# Ancilla is 2 qubits (q0, q1), so |q1 q0 = 00> corresponds to every 4th index (0, 4, 8, 12)
A_block_encoded = U_full[0::4, 0::4]

print("\nBlock-Encoded Matrix U[0::4, 0::4] * lambda (Recovered A_poisson):")
print(np.round(A_block_encoded * lam, 4).real)

print("\nMax error || lambda * U_block - A_poisson ||:")
print(np.linalg.norm(A_block_encoded * lam - A_poisson))

Automatic Pauli LCU Decomposition of 1D Poisson Matrix:
  +2.00 * II
  -1.00 * IX
  -0.50 * XX
  -0.50 * YY

1-Norm (lambda): 4.0

LCU Circuit for 4x4 Poisson Matrix:
     ┌────────────────────────────────────┐                        ┌─────────────────────────────────────┐
q_0: ┤0                                   ├───■───────o───────■────┤0                                    ├
     │  PREP(0.70711,0.5,0.35355,0.35355) │   │       │       │    │  PREP†(0.70711,0.5,0.35355,0.35355) │
q_1: ┤1                                   ├───o───────■───────■────┤1                                    ├
     └────────────────────────────────────┘┌──┴───┐┌──┴───┐┌──┴───┐└─────────────────────────────────────┘
q_2: ──────────────────────────────────────┤0     ├┤0     ├┤0     ├───────────────────────────────────────
                                           │  -IX ││  -XX ││  -YY │                                       
q_3: ──────────────────────────────────────┤1     ├┤1     ├┤1     ├─────────────────